# ⚽ Pipeline ETL — Campeonato Brasileiro Série A (2003–2026)

**Objetivo:** construir um pipeline de dados de ponta a ponta:

| Etapa | O que acontece | Saída |
|---|---|---|
| **1. Extract** | Ingestão do histórico 2003–2024 (GitHub) e dos dados recentes 2025–2026 (football-data.co.uk) | `data/raw/` (camada *bronze*) |
| **2. Profiling** | Diagnóstico de qualidade: nulos, duplicidades, inconsistências | relatório no notebook |
| **3. Transform** | Padronização de schema, tipos, temporada, resultado, estádios, nomes de clubes | DataFrame limpo (camada *silver*) |
| **4. Union** | Junta histórico + recente no mesmo schema e valida | `partidas` |
| **5. Modelagem** | Esquema estrela: `fato_partida` + dimensões | `output/*.csv` (camada *gold*) |
| **6. Load** | Carga no MySQL | banco `brasileirao` |

> Rode as células **em ordem**. As células de validação usam `assert`: se algo quebrar, o pipeline para e mostra o motivo — isso é proposital (*fail fast*).

In [ ]:
# Se estiver no Colab, só é preciso instalar as libs de banco (pandas já vem instalado)
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q sqlalchemy pymysql

In [ ]:
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)

# ============================ CONFIGURAÇÃO ============================
BASE_DIR = Path("/content/brasileirao_etl") if IN_COLAB else Path.cwd().parent
RAW_DIR  = BASE_DIR / "data" / "raw"
OUT_DIR  = BASE_DIR / "output"
SQL_DIR  = BASE_DIR / "sql"
for d in (RAW_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

URL_HISTORICO = "https://raw.githubusercontent.com/adaoduque/Brasileirao_Dataset/master/campeonato-brasileiro-full.csv"
URL_RECENTE   = "https://www.football-data.co.uk/new/BRA.csv"   # Série A 2012→hoje, atualizado semanalmente

ANO_INICIO_RECENTE = 2025      # a partir de qual temporada usar a fonte recente
TEMPORADA_ATUAL    = 2026      # temporada em andamento (marcada como em_andamento no banco)

print("Ambiente:", "Google Colab" if IN_COLAB else "Local")
print("Diretório base:", BASE_DIR)

## 1. Extract — dados históricos (2003–2024)

Baixa o CSV e guarda uma cópia **bruta e imutável** em `data/raw/`. Se o download falhar, usa a cópia local (útil para trabalhar offline e para reprodutibilidade).

In [ ]:
def extrair_csv(url: str, arquivo_local: Path, **read_kwargs) -> pd.DataFrame:
    try:
        df = pd.read_csv(url, **read_kwargs)
        df.to_csv(arquivo_local, index=False)
        print(f"✅ Download OK: {url}")
    except Exception as e:
        print(f"⚠️ Falha no download ({type(e).__name__}: {e}). Usando cópia local {arquivo_local.name}")
        df = pd.read_csv(arquivo_local, **read_kwargs)
    print(f"   {df.shape[0]:,} linhas × {df.shape[1]} colunas")
    return df

df_hist_raw = extrair_csv(URL_HISTORICO, RAW_DIR / "brasileirao_raw_2003_2024.csv")
df_hist_raw.head()

## 2. Profiling — diagnóstico de qualidade

Antes de limpar, **medir**. Achados relevantes deste dataset (documente-os no README do portfólio — mostra senso crítico):

1. **Temporada ≠ ano civil**: a temporada 2020 (pandemia) terminou em fev/2021. Filtrar por ano da data mistura temporadas.
2. **`vencedor` inconsistente com o placar** em pelo menos 1 jogo → o resultado deve ser **recalculado** a partir dos gols.
3. **Estádios**: espaços extras, marcação `*(PF)` (portões fechados) no nome e várias grafias para o mesmo estádio (ex.: *Morumbi* / *MorumBIS*).
4. **Datas erradas**: em 3 rodadas (2008, 2009, 2010) a fonte repetiu a data de outra rodada — descoberto por um teste de consistência.
5. **2016 tem 379 jogos**: Chapecoense × Atlético-MG não foi disputado (tragédia aérea da Chapecoense).
6. `formacao_*` e `tecnico_*` só existem a partir de ~2014 (nulos esperados, não erro).
7. Coluna `rodata` com erro de digitação (→ `rodada`).

In [ ]:
def perfil(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "nulos": df.isna().sum(),
        "%_nulos": (df.isna().mean() * 100).round(1),
        "distintos": df.nunique(),
        "exemplo": df.iloc[0],
    })

display(perfil(df_hist_raw))

print("IDs duplicados:", df_hist_raw["ID"].duplicated().sum())
print("Partidas duplicadas (data+mandante+visitante):",
      df_hist_raw.duplicated(["data", "mandante", "visitante"]).sum())
print("Arenas com espaço extra:", (df_hist_raw["arena"] != df_hist_raw["arena"].str.strip()).sum())

anos = pd.to_datetime(df_hist_raw["data"], format="%d/%m/%Y").dt.year
print("\nPartidas por ANO CIVIL (repare 2020 e 2021):")
print(anos.value_counts().sort_index().to_dict())

## 3. Transform — histórico

### 3.1 Schema padronizado
Nomes em `snake_case`, tipos corretos e strings sem espaços nas pontas. Esse é o **schema canônico** que a fonte recente também vai seguir.

In [ ]:
SCHEMA_PADRAO = [
    "id_origem", "fonte", "ano_temporada", "rodada", "data_partida", "hora_partida",
    "mandante", "visitante", "gols_mandante", "gols_visitante",
    "formacao_mandante", "formacao_visitante", "tecnico_mandante", "tecnico_visitante",
    "arena", "mandante_uf", "visitante_uf", "vencedor_original",
]

df_hist = (
    df_hist_raw
    .rename(columns={
        "ID": "id_origem", "rodata": "rodada", "data": "data_partida", "hora": "hora_partida",
        "mandante_Placar": "gols_mandante", "visitante_Placar": "gols_visitante",
        "mandante_Estado": "mandante_uf", "visitante_Estado": "visitante_uf",
        "vencedor": "vencedor_original",
    })
    .assign(fonte="adaoduque/github")
)

# Strings: tira espaços das pontas e transforma "" em nulo
obj_cols = df_hist.select_dtypes(include=["object", "string"]).columns
df_hist[obj_cols] = df_hist[obj_cols].apply(lambda s: s.str.strip().replace("", pd.NA))

df_hist["data_partida"] = pd.to_datetime(df_hist["data_partida"], format="%d/%m/%Y")
df_hist["hora_partida"] = pd.to_datetime(df_hist["hora_partida"], format="%H:%M").dt.strftime("%H:%M:%S")

### 3.2 Temporada
Regra explícita (e documentada) em vez de "chute" pelo ano: jogos até 28/02/2021 pertencem à temporada **2020**.
Depois validamos a quantidade de jogos por temporada contra o formato oficial do campeonato.

In [ ]:
# (ano civil, data limite, temporada correta) — exceções de calendário
EXCECOES_TEMPORADA = [(2021, "2021-02-28", 2020)]

df_hist["ano_temporada"] = df_hist["data_partida"].dt.year
for ano, limite, temporada in EXCECOES_TEMPORADA:
    mask = (df_hist["data_partida"].dt.year == ano) & (df_hist["data_partida"] <= limite)
    df_hist.loc[mask, "ano_temporada"] = temporada

# Formato oficial: 24 clubes (2003-04), 22 (2005), 20 (2006+) → n*(n-1) jogos
JOGOS_ESPERADOS = {2003: 552, 2004: 552, 2005: 462, 2016: 379}  # 2016: 1 jogo não disputado
contagem = df_hist.groupby("ano_temporada").size()
divergentes = {a: n for a, n in contagem.items() if n != JOGOS_ESPERADOS.get(a, 380)}
assert not divergentes, f"Temporadas com nº de jogos inesperado: {divergentes}"
print("✅ Todas as temporadas com o número de jogos esperado")
print(contagem.to_dict())

### 3.3 Correção de datas erradas na fonte
O teste *"cada clube joga no máximo 1 vez por dia"* revelou que, em 3 rodadas, a fonte copiou a data do **domingo de uma rodada anterior** (ex.: rodada 11/2008 com a data da rodada 9).
Como os demais jogos dessas rodadas foram num sábado, a data correta inferida é o **domingo seguinte**. A correção é explícita, auditável e verificada logo abaixo.

In [ ]:
def conflitos_de_data(df: pd.DataFrame) -> pd.DataFrame:
    # Clubes escalados em mais de um jogo no mesmo dia (impossível na prática)
    longo = pd.concat([
        df[["ano_temporada", "rodada", "data_partida", "mandante"]].rename(columns={"mandante": "clube"}),
        df[["ano_temporada", "rodada", "data_partida", "visitante"]].rename(columns={"visitante": "clube"}),
    ])
    return longo[longo.duplicated(["data_partida", "clube"], keep=False)]

print("Conflitos antes:")
display(conflitos_de_data(df_hist).groupby(["ano_temporada", "rodada", "data_partida"]).size().rename("clubes"))

# (temporada, rodada, data errada) → data correta
CORRECOES_DATA = {
    (2008, 11, "2008-07-06"): "2008-07-13",
    (2009,  6, "2009-05-31"): "2009-06-14",
    (2010,  5, "2010-05-23"): "2010-05-30",
}
for (temp, rod, errada), certa in CORRECOES_DATA.items():
    mask = (df_hist["ano_temporada"] == temp) & (df_hist["rodada"] == rod) & (df_hist["data_partida"] == errada)
    df_hist.loc[mask, "data_partida"] = pd.Timestamp(certa)
    print(f"{temp} rodada {rod}: {mask.sum()} jogos  {errada} → {certa}")

assert conflitos_de_data(df_hist).empty, "Ainda há clubes com 2 jogos no mesmo dia"
print("✅ Nenhum conflito de data restante")

### 3.4 Resultado recalculado a partir do placar
Nunca confie numa coluna derivada se você tem a coluna-fonte. `resultado`: **M** (mandante venceu), **E** (empate), **V** (visitante venceu).

In [ ]:
def calcular_resultado(df: pd.DataFrame) -> pd.Series:
    return pd.Series(
        np.select([df["gols_mandante"] > df["gols_visitante"],
                   df["gols_mandante"] < df["gols_visitante"]], ["M", "V"], default="E"),
        index=df.index,
    )

df_hist["resultado"] = calcular_resultado(df_hist)

vencedor_calc = np.select(
    [df_hist["resultado"] == "M", df_hist["resultado"] == "V"],
    [df_hist["mandante"], df_hist["visitante"]], default="-")
inconsistentes = df_hist[vencedor_calc != df_hist["vencedor_original"]]
print(f"⚠️ {len(inconsistentes)} partida(s) com 'vencedor' divergente do placar (corrigidas):")
display(inconsistentes[["id_origem", "data_partida", "mandante", "gols_mandante",
                        "gols_visitante", "visitante", "vencedor_original", "resultado"]])

### 3.5 Estádios
1. Extrai a marcação de **portões fechados** (`*(PF)`) para uma coluna booleana.
2. Unifica grafias diferentes do mesmo estádio num nome canônico (dicionário de-para).

In [ ]:
RE_PF = re.compile(r"\s*\(?\*\s*\(?PF\)", flags=re.IGNORECASE)

df_hist["portoes_fechados"] = df_hist["arena"].str.contains(RE_PF).astype(int)
df_hist["arena"] = df_hist["arena"].str.replace(RE_PF, "", regex=True).str.strip()

# de-para: variação → nome canônico
ESTADIOS_ALIAS = {
    "A Campanella": "Anacleto Campanella", "A.Campanella": "Anacleto Campanella",
    "Adelmar da Costa Carvalho": "Ilha do Retiro",
    "Arena de Pernambuco": "Arena Pernambuco",
    "Boca do Jacaré": "Arena do Jacaré",
    "Estádio Joaquim Henrique Nogueira-Arena do jacaré": "Arena do Jacaré",
    "Itaipava Arena Fonte Nova": "Arena Fonte Nova",
    "Arena BRB Mané Garrincha": "Mané Garrincha", "Estádio Nacional de Brasília": "Mané Garrincha",
    "Estádio Joaquim Américo Guimarães": "Arena da Baixada", "Ligga Arena": "Arena da Baixada",
    "Kyocera Arena": "Arena da Baixada",
    "Beira Rio": "Beira-Rio", "Estádio Beira-Rio": "Beira-Rio", "Estádio José Pinheiro Borda": "Beira-Rio",
    "Estádio Brinco de Ouro": "Brinco de Ouro",
    "Bruno J Daniel": "Bruno José Daniel", "Bruno J.Daniel": "Bruno José Daniel",
    "Castelão": "Arena Castelão", "Castelão (CE)": "Arena Castelão", "Plácido Castelo": "Arena Castelão",
    "Centenário": "Centenário (Caxias do Sul)", "Centenário (RS)": "Centenário (Caxias do Sul)",
    "Estádio Francisco Stédile (Centenário)": "Centenário (Caxias do Sul)",
    "Estádio Francisco Stédile | Centenário": "Centenário (Caxias do Sul)",
    "Estádio Dr. Oswaldo Teixeira Duarte": "Canindé",
    "Engenhão": "Nilton Santos", "Olímpico Engenhão": "Nilton Santos", "Estádio Nilton Santos": "Nilton Santos",
    "Estádio Antônio Accioly - Atlético Goianiense": "Antônio Accioly", "Estádio Antônio Accioly": "Antônio Accioly",
    "Estádio Giulite Coutinho": "Giulite Coutinho", "G Coutinho": "Giulite Coutinho", "Édson Passos": "Giulite Coutinho",
    "Estádio Hailé Pinheiro - Serrinha": "Serrinha", "Estádio da Serrinha": "Serrinha",
    "Estádio Heriberto Hülse": "Heriberto Hülse", "Heriberto Hulse": "Heriberto Hülse",
    "Estádio Independência": "Independência", "Estádio Raimundo Sampaio": "Independência",
    "Estádio Ipatingão": "Ipatingão",
    "Estádio Municipal General Raulino de Oliveira": "Raulino de Oliveira", "R de Oliveira": "Raulino de Oliveira",
    "Estádio Municipal Juscelino Kubitschek": "Juscelino Kubitschek", "Juscelino Kubitscheck": "Juscelino Kubitschek",
    "Estádio Municipal Parque do Sabiá": "Parque do Sabiá",
    "Estádio Municipal Paulo Machado de Carvalho": "Pacaembu",
    "Estádio Olímpico Pedro Ludovico Teixeira": "Olímpico Pedro Ludovico",
    "Olímpico": "Olímpico Monumental",
    "Estádio Presidente Vargas": "Presidente Vargas", "Pres Vargas": "Presidente Vargas",
    "Estádio Serra Dourada": "Serra Dourada",
    "Estádio do Governo do Estado de Goiás (Serra Dourada)": "Serra Dourada",
    "Estádio São Januário": "São Januário", "Estádio Vasco da Gama": "São Januário",
    "Estádio Urbano Caldeira": "Vila Belmiro",
    "Estádio Willie Davids": "Willie Davids",
    "Estádio de Pituaçu": "Pituaçu",
    "Estádio do Arruda": "Arruda",
    "Luso Brasileiro": "Luso-Brasileiro",
    "MorumBIS": "Morumbi",
    "Parque Antártica": "Palestra Itália",
    "Juiz de Fora": "Mário Helênio", "Municipal Juiz de Fora": "Mário Helênio", "Helenão": "Mário Helênio",
    "Estádio Alberto Oliveira": "Jóia da Princesa",
    "Cláudio Moacyr": "Moacyrzão",
    "Romildo Ferreira": "Romildão",
    "Estádio Doutor Adhemar de Barros": "Fonte Luminosa",
    "Eduardo José Farah": "Prudentão",
    "Benedito Teixeira": "Teixeirão",
    "Morenão": "Pedro Pedrossian",
    "Papa J.Paulo II": "Papa João Paulo II",
}

n_antes = df_hist["arena"].nunique()
df_hist["arena"] = df_hist["arena"].replace(ESTADIOS_ALIAS)
print(f"Estádios distintos: {n_antes} → {df_hist['arena'].nunique()}")
print(f"Jogos com portões fechados: {df_hist['portoes_fechados'].sum()}")

## 4. Extract + Transform — dados recentes (2025–2026)

**Fonte:** [football-data.co.uk](https://www.football-data.co.uk/brazil.php) — CSV gratuito e estável com a Série A desde 2012, atualizado semanalmente.

Diferenças em relação ao histórico (e como tratamos):

| Problema | Tratamento |
|---|---|
| Nomes dos clubes diferentes (`Flamengo RJ`, `Sport Recife`…) | normalização + dicionário de-para, com **validação que para o pipeline** se surgir um nome desconhecido |
| Não tem estado (UF) | busca a UF na própria base histórica; clubes estreantes vêm de um dicionário manual |
| Não tem rodada, estádio, técnico, formação | ficam `NULL` (documentado) |

> 💡 **Bônus para o portfólio:** os anos 2012–2024 existem nas **duas** fontes. Usamos essa sobreposição para validar o de-para de nomes e conferir se os placares batem (*reconciliação entre fontes*).

In [ ]:
df_rec_raw = extrair_csv(URL_RECENTE, RAW_DIR / "football_data_BRA.csv")
display(df_rec_raw.head())
print("Temporadas disponíveis:", sorted(df_rec_raw["Season"].unique()))

In [ ]:
def chave(nome: str) -> str:
    # 'Atlético-MG' → 'atleticomg' (sem acento, minúsculo, só letras/números)
    s = unicodedata.normalize("NFKD", str(nome)).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]", "", s.lower())

# Base de clubes conhecidos (nome padrão → UF), vinda do histórico
UF_CLUBE = (pd.concat([
    df_hist[["mandante", "mandante_uf"]].set_axis(["clube", "uf"], axis=1),
    df_hist[["visitante", "visitante_uf"]].set_axis(["clube", "uf"], axis=1),
]).drop_duplicates().set_index("clube")["uf"].to_dict())

# Clubes que nunca jogaram a Série A entre 2003-2024 (acrescente aqui se aparecer um novo)
CLUBES_NOVOS_UF = {"Mirassol": "SP", "Remo": "PA", "Novorizontino": "SP"}
UF_CLUBE.update(CLUBES_NOVOS_UF)

# De-para explícito: chave(nome na fonte recente) → nome padrão
ALIAS_CLUBES = {
    "flamengorj": "Flamengo", "botafogo": "Botafogo-RJ", "botafogorj": "Botafogo-RJ",
    "sportrecife": "Sport", "chapecoensesc": "Chapecoense", "vascodagama": "Vasco",
    "rbbragantino": "Bragantino", "redbullbragantino": "Bragantino",
    "atleticopr": "Athletico-PR", "atleticoparanaense": "Athletico-PR",
    "atleticomineiro": "Atletico-MG", "americamineiro": "America-MG", "atleticogoianiense": "Atletico-GO",
    "gremionovorizontino": "Novorizontino",
}
MAPA_CHAVE = {chave(c): c for c in UF_CLUBE} | ALIAS_CLUBES

nomes_fonte = pd.unique(df_rec_raw[["Home", "Away"]].values.ravel())
de_para = {n: MAPA_CHAVE.get(chave(n)) for n in nomes_fonte}

# Obrigatório mapear 100% das temporadas que vamos carregar; nas antigas (só reconciliação) apenas avisamos
recentes = df_rec_raw[df_rec_raw["Season"] >= ANO_INICIO_RECENTE]
nomes_recentes = set(recentes["Home"]) | set(recentes["Away"])
avisos = sorted(n for n, p in de_para.items() if p is None and n not in nomes_recentes)
if avisos:
    print("ℹ️ Sem de-para (só temporadas antigas, ficam fora da reconciliação):", avisos)
nao_mapeados = sorted(n for n in nomes_recentes if de_para[n] is None)
assert not nao_mapeados, (
    f"Clubes sem de-para: {nao_mapeados}\n"
    "→ adicione em ALIAS_CLUBES (se já existe no histórico) ou em CLUBES_NOVOS_UF (se é estreante).")
print("✅ Todos os clubes mapeados:")
print({k: v for k, v in sorted(de_para.items()) if k != v})

In [ ]:
# Reconciliação: nos anos em comum, as duas fontes concordam?
FUSO_FONTE_RECENTE = "Europe/London"   # football-data publica horários do Reino Unido

# A fonte traz espaços sobrando em Date/Time → strip antes de converter
rec_date_clean = df_rec_raw["Date"].str.strip()
rec_time_clean = df_rec_raw["Time"].str.strip()

rec = df_rec_raw.assign(
    ano_temporada=df_rec_raw["Season"].astype(int),
    data_partida=pd.to_datetime(rec_date_clean, dayfirst=True),
    mandante=df_rec_raw["Home"].map(de_para),
    visitante=df_rec_raw["Away"].map(de_para),
    hora_original=pd.to_datetime(rec_time_clean, format="%H:%M").dt.strftime("%H:%M:%S"),
)
dt_brasilia = (
    pd.to_datetime(rec_date_clean + " " + rec_time_clean, dayfirst=True)
      .dt.tz_localize(FUSO_FONTE_RECENTE, ambiguous="NaT", nonexistent="shift_forward")
      .dt.tz_convert("America/Sao_Paulo").dt.tz_localize(None)
)
# jogo às 23h em Brasília = 03h do dia seguinte em Londres → data E hora mudam
rec["data_brasilia"] = dt_brasilia.dt.normalize().fillna(rec["data_partida"])
rec["hora_brasilia"] = dt_brasilia.dt.strftime("%H:%M:%S")
comum = df_hist.merge(rec, on=["ano_temporada", "mandante", "visitante"], suffixes=("", "_rec"))
if len(comum):
    diverg = comum[(comum["gols_mandante"] != comum["HG"]) | (comum["gols_visitante"] != comum["AG"])]
    print(f"Jogos casados entre as fontes: {len(comum):,} | placares divergentes: {len(diverg)}")

    # Qual interpretação de horário bate com o histórico (que está em horário de Brasília)?
    pct_conv = (comum["hora_partida"] == comum["hora_brasilia"]).mean()
    pct_orig = (comum["hora_partida"] == comum["hora_original"]).mean()
    CONVERTER_FUSO = pct_conv >= pct_orig
    print(f"Horário igual ao histórico: convertido {pct_conv:.0%} × sem converter {pct_orig:.0%} "
          f"→ {'convertendo' if CONVERTER_FUSO else 'mantendo'} o horário da fonte")
    data_rec = comum["data_brasilia"] if CONVERTER_FUSO else comum["data_partida_rec"]
    print(f"Datas diferentes entre as fontes (jogos remarcados?): {(comum['data_partida'] != data_rec).sum()}")
    display(diverg[["ano_temporada", "data_partida", "mandante", "visitante",
                    "gols_mandante", "gols_visitante", "HG", "AG"]].head(10))
else:
    CONVERTER_FUSO = True
    print("Sem temporadas em comum para reconciliar.")

In [ ]:
df_rec = (
    rec[rec["Season"] >= ANO_INICIO_RECENTE]
    .dropna(subset=["HG", "AG"])                         # descarta jogos ainda não disputados
    .assign(
        id_origem=pd.NA,
        fonte="football-data.co.uk",
        rodada=pd.NA,
        data_partida=lambda d: d["data_brasilia"] if CONVERTER_FUSO else d["data_partida"],
        hora_partida=lambda d: d["hora_brasilia"] if CONVERTER_FUSO else d["hora_original"],
        gols_mandante=lambda d: d["HG"].astype(int),
        gols_visitante=lambda d: d["AG"].astype(int),
        mandante_uf=lambda d: d["mandante"].map(UF_CLUBE),
        visitante_uf=lambda d: d["visitante"].map(UF_CLUBE),
        portoes_fechados=0,
    )
)
for col in ["formacao_mandante", "formacao_visitante", "tecnico_mandante",
            "tecnico_visitante", "arena", "vencedor_original"]:
    df_rec[col] = pd.NA

df_rec["resultado"] = calcular_resultado(df_rec)

# Sanidade: o 'Res' (H/D/A) da fonte bate com o placar?
assert (df_rec["Res"].map({"H": "M", "D": "E", "A": "V"}) == df_rec["resultado"]).all()
print(f"✅ {len(df_rec)} partidas recentes:", df_rec.groupby("ano_temporada").size().to_dict())

## 5. Union + validação final

In [ ]:
COLUNAS = SCHEMA_PADRAO + ["resultado", "portoes_fechados"]

partidas = (
    pd.concat([df_hist[COLUNAS], df_rec[COLUNAS]], ignore_index=True)
    .drop_duplicates(subset=["data_partida", "mandante", "visitante"], keep="first")
    .sort_values(["ano_temporada", "data_partida", "hora_partida", "mandante"])
    .reset_index(drop=True)
)
partidas.insert(0, "id_partida", np.arange(1, len(partidas) + 1))

# ---- Testes de qualidade (data quality checks) ----
checks = {
    "sem partidas duplicadas":        not partidas.duplicated(["data_partida", "mandante", "visitante"]).any(),
    "mandante ≠ visitante":           (partidas["mandante"] != partidas["visitante"]).all(),
    "gols não negativos":             (partidas[["gols_mandante", "gols_visitante"]] >= 0).all().all(),
    "UF preenchida":                  partidas[["mandante_uf", "visitante_uf"]].notna().all().all(),
    "temporadas contínuas":           set(range(2003, partidas["ano_temporada"].max() + 1)) == set(partidas["ano_temporada"]),
    "cada clube joga 1x por data":    not pd.concat([partidas[["data_partida", "mandante"]].set_axis(["d", "c"], axis=1),
                                                     partidas[["data_partida", "visitante"]].set_axis(["d", "c"], axis=1)]
                                                    ).duplicated().any(),
}
for nome, ok in checks.items():
    print("✅" if ok else "❌", nome)
assert all(checks.values()), "Falha nos testes de qualidade"

print(f"\nTotal: {len(partidas):,} partidas | {partidas['ano_temporada'].min()}–{partidas['ano_temporada'].max()}")
partidas.groupby(["ano_temporada", "fonte"]).size().unstack(fill_value=0).tail(6)

## 6. Modelagem dimensional (esquema estrela)

```
                 dim_temporada
                       │
 dim_clube ──┬── fato_partida ──── dim_estadio
 (mandante)  │         │
 dim_clube ──┘   (1 linha por jogo)
 (visitante)
     │
 dim_estado
```

- `dim_clube` é uma **role-playing dimension** (usada 2× na fato: mandante e visitante).
- No banco também criamos as views `vw_partida_clube` (1 linha por clube por jogo) e `vw_classificacao` — elas simplificam muito o Power BI.

In [ ]:
NOME_EXIBICAO = {
    "America-MG": "América-MG", "America-RN": "América-RN", "Atletico-GO": "Atlético-GO",
    "Atletico-MG": "Atlético-MG", "Avai": "Avaí", "Botafogo-RJ": "Botafogo", "Ceara": "Ceará",
    "Criciuma": "Criciúma", "Cuiaba": "Cuiabá", "Goias": "Goiás", "Gremio": "Grêmio",
    "Gremio Prudente": "Grêmio Prudente", "Nautico": "Náutico", "Parana": "Paraná",
    "Santo Andre": "Santo André", "Sao Caetano": "São Caetano", "Sao Paulo": "São Paulo",
    "Vitoria": "Vitória",
}

ESTADOS = [
    ("AC","Acre","Norte"),("AL","Alagoas","Nordeste"),("AP","Amapá","Norte"),("AM","Amazonas","Norte"),
    ("BA","Bahia","Nordeste"),("CE","Ceará","Nordeste"),("DF","Distrito Federal","Centro-Oeste"),
    ("ES","Espírito Santo","Sudeste"),("GO","Goiás","Centro-Oeste"),("MA","Maranhão","Nordeste"),
    ("MT","Mato Grosso","Centro-Oeste"),("MS","Mato Grosso do Sul","Centro-Oeste"),
    ("MG","Minas Gerais","Sudeste"),("PA","Pará","Norte"),("PB","Paraíba","Nordeste"),
    ("PR","Paraná","Sul"),("PE","Pernambuco","Nordeste"),("PI","Piauí","Nordeste"),
    ("RJ","Rio de Janeiro","Sudeste"),("RN","Rio Grande do Norte","Nordeste"),
    ("RS","Rio Grande do Sul","Sul"),("RO","Rondônia","Norte"),("RR","Roraima","Norte"),
    ("SC","Santa Catarina","Sul"),("SP","São Paulo","Sudeste"),("SE","Sergipe","Nordeste"),
    ("TO","Tocantins","Norte"),
]
dim_estado = pd.DataFrame(ESTADOS, columns=["sigla_uf", "nome_uf", "regiao"])

# --- dim_clube ---
clubes = sorted(set(partidas["mandante"]) | set(partidas["visitante"]))
dim_clube = pd.DataFrame({"id_clube": range(1, len(clubes) + 1), "nome_clube": clubes})
dim_clube["nome_exibicao"] = dim_clube["nome_clube"].map(NOME_EXIBICAO).fillna(dim_clube["nome_clube"])
dim_clube["sigla_uf"] = dim_clube["nome_clube"].map(UF_CLUBE)

# --- dim_estadio ---
estadios = sorted(partidas["arena"].dropna().unique())
dim_estadio = pd.DataFrame({"id_estadio": range(1, len(estadios) + 1), "nome_estadio": estadios})

# --- dim_temporada ---
times_por_temp = pd.concat([partidas[["ano_temporada", "mandante"]].set_axis(["ano_temporada", "clube"], axis=1),
                            partidas[["ano_temporada", "visitante"]].set_axis(["ano_temporada", "clube"], axis=1)])
dim_temporada = (
    partidas.groupby("ano_temporada")
    .agg(qtd_rodadas=("rodada", "max"), qtd_partidas=("id_partida", "size"),
         data_inicio=("data_partida", "min"), data_fim=("data_partida", "max"), fonte=("fonte", "first"))
    .join(times_por_temp.groupby("ano_temporada")["clube"].nunique().rename("qtd_clubes"))
    .reset_index()
)
dim_temporada["em_andamento"] = (dim_temporada["ano_temporada"] == TEMPORADA_ATUAL).astype(int)
dim_temporada = dim_temporada[["ano_temporada", "qtd_clubes", "qtd_rodadas", "qtd_partidas",
                               "data_inicio", "data_fim", "fonte", "em_andamento"]]

# --- fato_partida ---
id_clube = dim_clube.set_index("nome_clube")["id_clube"]
id_estadio = dim_estadio.set_index("nome_estadio")["id_estadio"]
fato_partida = partidas.assign(
    id_clube_mandante=partidas["mandante"].map(id_clube),
    id_clube_visitante=partidas["visitante"].map(id_clube),
    id_estadio=partidas["arena"].map(id_estadio).astype("Int64"),
    rodada=partidas["rodada"].astype("Int64"),
    id_origem=partidas["id_origem"].astype("Int64"),
)[[
    "id_partida", "id_origem", "fonte", "ano_temporada", "rodada", "data_partida", "hora_partida",
    "id_clube_mandante", "id_clube_visitante", "id_estadio", "gols_mandante", "gols_visitante",
    "resultado", "formacao_mandante", "formacao_visitante", "tecnico_mandante", "tecnico_visitante",
    "portoes_fechados",
]]

# Integridade referencial antes de mandar pro banco
assert dim_clube["sigla_uf"].isin(dim_estado["sigla_uf"]).all()
assert fato_partida[["id_clube_mandante", "id_clube_visitante"]].notna().all().all()

TABELAS = {  # ordem de carga respeita as FKs
    "dim_estado": dim_estado, "dim_clube": dim_clube, "dim_estadio": dim_estadio,
    "dim_temporada": dim_temporada, "fato_partida": fato_partida,
}
for nome, t in TABELAS.items():
    print(f"{nome:15s} {len(t):>6,} linhas")

In [ ]:
# Exporta a camada gold em CSV (backup + alternativa de carga via Workbench)
for nome, t in TABELAS.items():
    t.to_csv(OUT_DIR / f"{nome}.csv", index=False, na_rep=r"\N", date_format="%Y-%m-%d", lineterminator="\n")
print("✅ CSVs gerados em", OUT_DIR)

if IN_COLAB:  # baixa tudo num .zip para levar ao MySQL local
    import shutil
    from google.colab import files
    shutil.make_archive("/content/brasileirao_gold", "zip", OUT_DIR)
    files.download("/content/brasileirao_gold.zip")

## 7. Load — MySQL

O **Colab roda na nuvem e não enxerga o MySQL do seu computador** (`localhost`). Escolha um caminho:

**Opção A — rodar este notebook localmente** (Jupyter / VS Code) com o MySQL ligado → a célula abaixo cria o schema, carrega tudo e cria as views automaticamente.

**Opção B — Colab + Workbench:** baixe o `.zip` gerado acima, extraia em `output/` e rode no Workbench, em ordem: `sql/01_schema.sql` → `sql/03_load_csv.sql` → `sql/02_views.sql`.

> 🔐 Nunca deixe senha escrita no notebook que vai pro GitHub. Use variável de ambiente (`MYSQL_PASSWORD`) ou `getpass`.

In [ ]:
import os
from getpass import getpass

CARREGAR_NO_MYSQL = not IN_COLAB   # mude para True/False se quiser forçar

if CARREGAR_NO_MYSQL:
    from sqlalchemy import create_engine, text
    from sqlalchemy.engine import URL

    url = URL.create(
        "mysql+pymysql",
        username=os.getenv("MYSQL_USER", "root"),
        password=os.getenv("MYSQL_PASSWORD") or getpass("Senha do MySQL: "),
        host=os.getenv("MYSQL_HOST", "localhost"),
        port=int(os.getenv("MYSQL_PORT", 3306)),
        query={"charset": "utf8mb4"},
    )
    engine = create_engine(url)

    def executar_script(arquivo: Path):
        sql = re.sub(r"--[^\n]*", "", arquivo.read_text(encoding="utf-8"))
        with engine.begin() as conn:
            for stmt in filter(str.strip, sql.split(";")):
                conn.execute(text(stmt))

    executar_script(SQL_DIR / "01_schema.sql")          # cria banco + tabelas (full refresh)
    engine_db = create_engine(url.set(database="brasileirao"))
    for nome, t in TABELAS.items():
        t.to_sql(nome, engine_db, if_exists="append", index=False, chunksize=1000, method="multi")
        print(f"✅ {nome}: {len(t):,} linhas carregadas")
    executar_script(SQL_DIR / "02_views.sql")

    # Verificação pós-carga: banco × DataFrame
    with engine_db.connect() as conn:
        for nome, t in TABELAS.items():
            n = conn.execute(text(f"SELECT COUNT(*) FROM {nome}")).scalar()
            assert n == len(t), f"{nome}: banco={n} df={len(t)}"
    print("✅ Contagens conferidas")
else:
    print("Carga no MySQL desativada (use a Opção B).")

In [ ]:
# Prova real: campeões de cada temporada a partir da view
if CARREGAR_NO_MYSQL:
    campeoes = pd.read_sql('''
        SELECT ano_temporada, clube, pontos, vitorias, saldo_gols
        FROM vw_classificacao WHERE posicao = 1 ORDER BY ano_temporada
    ''', engine_db)
    display(campeoes)

## 8. Próximos passos
- **Power BI:** conectar no banco `brasileirao` (ver `README.md` → seção Power BI) e importar `fato_partida`, as dimensões e as views.
- **Rodada para 2025–2026:** a fonte recente não traz a rodada; uma melhoria é derivá-la ou trocar por uma fonte que a tenha.
- **Automação:** transformar as células em funções num `etl.py` e agendar a atualização semanal da temporada em andamento.